In [10]:
import pandas as pd

# 1. Define our file paths
input_file = r'C:\Users\ASROCK\Desktop\Management\PROJECT\PROJECT 1 - AUTOMATED FINANCIAL PIPELINE\system-reconciliation-pipeline\data\raw\paysim_raw.csv'
output_file = r'C:\Users\ASROCK\Desktop\Management\PROJECT\PROJECT 1 - AUTOMATED FINANCIAL PIPELINE\system-reconciliation-pipeline\data\raw\system_a_golden.csv'

print("Loading the massive Kaggle dataset. This might take a few seconds...")

# 2. Load the data using pandas
df_full = pd.read_csv(input_file)
print(f"Original dataset size: {df_full.shape[0]:,} rows.")

# 3. Extract a random sample of 200,000 rows
print("Extracting a random sample of 200,000 rows...")
# We use random_state=42 so that if you run this again, it grabs the exact same 200k rows
df_sample = df_full.sample(n=200000, random_state=42)

# 4. Save this as our "System A" (The perfectly clean internal ledger)
df_sample.to_csv(output_file, index=False)
print(f"Success! System A saved to {output_file} with {df_sample.shape[0]:,} rows.")

# 5. Let's look at the first 5 rows to make sure it looks like real financial data
df_sample.head()

Loading the massive Kaggle dataset. This might take a few seconds...
Original dataset size: 6,362,620 rows.
Extracting a random sample of 200,000 rows...
Success! System A saved to C:\Users\ASROCK\Desktop\Management\PROJECT\PROJECT 1 - AUTOMATED FINANCIAL PIPELINE\system-reconciliation-pipeline\data\raw\system_a_golden.csv with 200,000 rows.


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
3737323,278,CASH_IN,330218.42,C632336343,20866.00,351084.42,C834976624,452419.57,122201.15,0,0
264914,15,PAYMENT,11647.08,C1264712553,30370.00,18722.92,M215391829,0.00,0.00,0,0
85647,10,CASH_IN,152264.21,C1746846248,106589.00,258853.21,C1607284477,201303.01,49038.80,0,0
5899326,403,TRANSFER,1551760.63,C333676753,0.00,0.00,C1564353608,3198359.45,4750120.08,0,0
2544263,206,CASH_IN,78172.30,C813403091,2921331.58,2999503.88,C1091768874,415821.90,337649.60,0,0


In [11]:
import pandas as pd
import numpy as np

# 1. Define our paths 
input_file = r'C:\Users\ASROCK\Desktop\Management\PROJECT\PROJECT 1 - AUTOMATED FINANCIAL PIPELINE\system-reconciliation-pipeline\data\raw\system_a_golden.csv'
output_file = r'C:\Users\ASROCK\Desktop\Management\PROJECT\PROJECT 1 - AUTOMATED FINANCIAL PIPELINE\system-reconciliation-pipeline\data\raw\system_b_messy.csv'

# 2. Load the System
df_b = pd.read_csv(input_file)
print(f"Loaded System A with {df_b.shape[0]:,} rows. Let's make a mess...")

# Set a random seed so our random errors are exactly the same every time we run this
np.random.seed(42)

# --- ERROR INJECTION 1: Missing Records (The Bank lost some transactions) ---
# We randomly drop 1% of the rows entirely
drop_indices = df_b.sample(frac=0.01).index
df_b = df_b.drop(drop_indices)
print(f"Dropped 1% of rows. System B now has {df_b.shape[0]:,} rows.")

# --- ERROR INJECTION 2: Amount Discrepancies (System rounding errors) ---
# We grab 2% of the rows and add a random cent value between -$0.50 and +$0.50
amount_error_indices = df_b.sample(frac=0.02).index
df_b.loc[amount_error_indices, 'amount'] += np.random.uniform(-0.50, 0.50, size=len(amount_error_indices))
df_b['amount'] = df_b['amount'].round(2) # Keep it to 2 decimal places like real money
print("Injected minor amount discrepancies (pennies) into 2% of records.")

# --- ERROR INJECTION 3: Text Typos (Human data entry errors) ---
# We grab another 2% of rows and mess up the Destination Name 
text_error_indices = df_b.sample(frac=0.02).index
df_b.loc[text_error_indices, 'nameDest'] = df_b.loc[text_error_indices, 'nameDest'].astype(str) + "_TYPO"
print("Injected text typos into 2% of destination names.")

# 3. Shuffle the deck! 
# We shuffle the order of the rows so System B isn't in the exact same order as System A
df_b = df_b.sample(frac=1).reset_index(drop=True)

# 4. Save the messy dataset
df_b.to_csv(output_file, index=False)
print(f"\nSuccess! System B (The Messy Bank Statement) saved to {output_file}")

Loaded System A with 200,000 rows. Let's make a mess...
Dropped 1% of rows. System B now has 198,000 rows.
Injected minor amount discrepancies (pennies) into 2% of records.
Injected text typos into 2% of destination names.

Success! System B (The Messy Bank Statement) saved to C:\Users\ASROCK\Desktop\Management\PROJECT\PROJECT 1 - AUTOMATED FINANCIAL PIPELINE\system-reconciliation-pipeline\data\raw\system_b_messy.csv


In [12]:
#Checking the data is already messy enough to build the pipeline
import pandas as pd

# 1. Define our paths
file_a = r'C:\Users\ASROCK\Desktop\Management\PROJECT\PROJECT 1 - AUTOMATED FINANCIAL PIPELINE\system-reconciliation-pipeline\data\raw\system_a_golden.csv'
file_b = r'C:\Users\ASROCK\Desktop\Management\PROJECT\PROJECT 1 - AUTOMATED FINANCIAL PIPELINE\system-reconciliation-pipeline\data\raw\system_b_messy.csv'

# 2. Load both datasets
print("Loading both systems for comparison...")
df_a = pd.read_csv(file_a)
df_b = pd.read_csv(file_b)

# 3. Print the High-Level Metrics
print("\n--- 📊 SANITY CHECK RESULTS ---")
print(f"System A (Golden) Total Rows: {len(df_a):,}")
print(f"System B (Messy) Total Rows:  {len(df_b):,}")
print(f"Difference (Missing Rows):    {len(df_a) - len(df_b):,}")

# 4. Prove the typos exist
# We are asking pandas to hunt down any row where the destination name contains our "_TYPO" tag
typos = df_b[df_b['nameDest'].astype(str).str.contains("_TYPO", na=False)]
print(f"\nFound {len(typos):,} records with text typos!")

# 5. Show the evidence
print("\nHere is a sneak peek at the corrupted text data our pipeline will have to fix:")
display(typos[['step', 'type', 'amount', 'nameDest']].head(3))

Loading both systems for comparison...

--- 📊 SANITY CHECK RESULTS ---
System A (Golden) Total Rows: 200,000
System B (Messy) Total Rows:  198,000
Difference (Missing Rows):    2,000

Found 3,960 records with text typos!

Here is a sneak peek at the corrupted text data our pipeline will have to fix:


,step,type,amount,nameDest
11,210,PAYMENT,14683.67,M1131995326_TYPO
67,205,TRANSFER,358802.61,C1136324849_TYPO
178,23,PAYMENT,4867.73,M227052453_TYPO


In [13]:
import pandas as pd

# 1. Define our File Paths
file_a = r'C:\Users\ASROCK\Desktop\Management\PROJECT\PROJECT 1 - AUTOMATED FINANCIAL PIPELINE\system-reconciliation-pipeline\data\raw\system_a_golden.csv'
file_b = r'C:\Users\ASROCK\Desktop\Management\PROJECT\PROJECT 1 - AUTOMATED FINANCIAL PIPELINE\system-reconciliation-pipeline\data\raw\system_b_messy.csv'

# Output paths for the leftovers
out_unmatched_a = r'C:\Users\ASROCK\Desktop\Management\PROJECT\PROJECT 1 - AUTOMATED FINANCIAL PIPELINE\system-reconciliation-pipeline\data\raw\unmatched_A.csv'
out_unmatched_b = r'C:\Users\ASROCK\Desktop\Management\PROJECT\PROJECT 1 - AUTOMATED FINANCIAL PIPELINE\system-reconciliation-pipeline\data\raw\unmatched_B.csv'

print("Loading datasets for Phase 2: Exact Matching...")
df_a = pd.read_csv(file_a)
df_b = pd.read_csv(file_b)

# 2. The Exact Match Logic (The "Easy" Reconciliations)
# We use a pandas 'outer merge' to compare every single column. 
# indicator=True is a brilliant trick: it creates a new column called '_merge' 
# that tells us exactly which file the row came from!
print("Running strict 1-to-1 reconciliation engine...")
reconciliation_df = pd.merge(df_a, df_b, how='outer', indicator=True)

# 3. Split the data into our three buckets
# Bucket 1: Perfect Matches (These are fully reconciled!)
perfect_matches = reconciliation_df[reconciliation_df['_merge'] == 'both'].copy()

# Bucket 2: Unmatched in System A (The true record, but System B lost or changed it)
unmatched_a = reconciliation_df[reconciliation_df['_merge'] == 'left_only'].copy()

# Bucket 3: Unmatched in System B (The corrupted records we injected)
unmatched_b = reconciliation_df[reconciliation_df['_merge'] == 'right_only'].copy()

# 4. Print the Results
print("\n--- 🏁 DAY 5 RECONCILIATION RESULTS ---")
print(f"✅ Reconciled (Perfect Matches): {len(perfect_matches):,} rows")
print(f"❌ Unmatched (Original System A Records): {len(unmatched_a):,} rows")
print(f"❌ Unmatched (Messy System B Records): {len(unmatched_b):,} rows")

# 5. Clean up the messy indicator column and save the unmatched records for Day 6
unmatched_a = unmatched_a.drop(columns=['_merge'])
unmatched_b = unmatched_b.drop(columns=['_merge'])

unmatched_a.to_csv(out_unmatched_a, index=False)
unmatched_b.to_csv(out_unmatched_b, index=False)

print("\nSaved the unmatched records to our data folder. Ready for complex matching tomorrow!")

Loading datasets for Phase 2: Exact Matching...
Running strict 1-to-1 reconciliation engine...

--- 🏁 DAY 5 RECONCILIATION RESULTS ---
✅ Reconciled (Perfect Matches): 190,200 rows
❌ Unmatched (Original System A Records): 9,800 rows
❌ Unmatched (Messy System B Records): 7,800 rows

Saved the unmatched records to our data folder. Ready for complex matching tomorrow!


In [14]:
import pandas as pd
import difflib

# 1. Load the Unmatched Leftovers from Day 5
file_unmatched_a = r'C:\Users\ASROCK\Desktop\Management\PROJECT\PROJECT 1 - AUTOMATED FINANCIAL PIPELINE\system-reconciliation-pipeline\data\raw\unmatched_A.csv'
file_unmatched_b = r'C:\Users\ASROCK\Desktop\Management\PROJECT\PROJECT 1 - AUTOMATED FINANCIAL PIPELINE\system-reconciliation-pipeline\data\raw\unmatched_B.csv'

print("Loading the unmatched records...")
df_un_a = pd.read_csv(file_unmatched_a)
df_un_b = pd.read_csv(file_unmatched_b)

# 2. The Smart Filter
# If we compare every row to every other row, it would take your laptop hours.
# Instead, we match them on columns we KNOW we didn't corrupt (step, type, nameOrig).
# This gives us pairs of transactions that are highly likely to be the same event.
print("Pairing up probable matches based on stable transaction keys...")
potentials = pd.merge(df_un_a, df_un_b, on=['step', 'type', 'nameOrig'], suffixes=('_A', '_B'))

# Isolate the records where the financial amount matches perfectly, 
# meaning the ONLY error must be a text typo in the destination name.
typo_candidates = potentials[potentials['amount_A'] == potentials['amount_B']].copy()

# 3. The Fuzzy Match Logic
def get_similarity(str1, str2):
    # Calculates how similar two strings are (0.0 to 1.0)
    return difflib.SequenceMatcher(None, str(str1), str(str2)).ratio()

print("Scanning for text typos using Sequence Matching algorithms...")
# Apply the algorithm to the Destination Name columns
typo_candidates['similarity_score'] = typo_candidates.apply(
    lambda row: get_similarity(row['nameDest_A'], row['nameDest_B']), axis=1
)

# 4. The Decision Rule
# We tell the system: "If the text is at least 80% similar, consider it a match!"
resolved_typos = typo_candidates[typo_candidates['similarity_score'] >= 0.80].copy()

# 5. Print Results
print("\n--- 🕵️‍♂️ DAY 6: FUZZY MATCHING RESULTS ---")
print(f"✅ Typo Errors Automatically Reconciled: {len(resolved_typos):,} rows")

print("\nEvidence of the typos our AI successfully caught:")
# We display the System A name, System B name, and the confidence score
display(resolved_typos[['nameDest_A', 'nameDest_B', 'similarity_score']].head())

Loading the unmatched records...
Pairing up probable matches based on stable transaction keys...
Scanning for text typos using Sequence Matching algorithms...

--- 🕵️‍♂️ DAY 6: FUZZY MATCHING RESULTS ---
✅ Typo Errors Automatically Reconciled: 3,701 rows

Evidence of the typos our AI successfully caught:


,nameDest_A,nameDest_B,similarity_score
0,C451111351,C451111351_TYPO,0.800000
1,M1979787155,M1979787155_TYPO,0.814815
3,C1446521801,C1446521801_TYPO,0.814815
6,C1725062057,C1725062057_TYPO,0.814815
7,C919978702,C919978702_TYPO,0.800000


In [15]:
import pandas as pd
import numpy as np

# 1. Load the Unmatched Leftovers from Day 5
file_unmatched_a = r'C:\Users\ASROCK\Desktop\Management\PROJECT\PROJECT 1 - AUTOMATED FINANCIAL PIPELINE\system-reconciliation-pipeline\data\raw\unmatched_A.csv'
file_unmatched_b = r'C:\Users\ASROCK\Desktop\Management\PROJECT\PROJECT 1 - AUTOMATED FINANCIAL PIPELINE\system-reconciliation-pipeline\data\raw\unmatched_B.csv'

print("Loading unmatched records for Numeric Tolerance matching...")
df_un_a = pd.read_csv(file_unmatched_a)
df_un_b = pd.read_csv(file_unmatched_b)

# 2. Pair them up based on everything EXCEPT the amount
# If the step, type, sender, and receiver all match, it's the same transaction.
print("Pairing up transactions to check for minor rounding errors...")
potentials = pd.merge(df_un_a, df_un_b, on=['step', 'type', 'nameOrig', 'nameDest'], suffixes=('_A', '_B'))

# 3. Calculate the Variance (The absolute difference between the two amounts)
potentials['variance'] = abs(potentials['amount_A'] - potentials['amount_B'])

# 4. The Decision Rule
# We only want rows where there IS a difference (> 0) but it is $0.50 or less.
tolerance_limit = 0.50
resolved_amounts = potentials[(potentials['variance'] > 0) & (potentials['variance'] <= tolerance_limit)].copy()

# 5. Print Results
print("\n--- ⚖️ DAY 7: NUMERIC VARIANCE RESULTS ---")
print(f"✅ Penny/Rounding Errors Automatically Reconciled: {len(resolved_amounts):,} rows")

print("\nEvidence of the numeric variances our pipeline successfully forgave:")
# We display the System A amount, System B amount, and the calculated difference
display(resolved_amounts[['amount_A', 'amount_B', 'variance']].head())

Loading unmatched records for Numeric Tolerance matching...
Pairing up transactions to check for minor rounding errors...

--- ⚖️ DAY 7: NUMERIC VARIANCE RESULTS ---
✅ Penny/Rounding Errors Automatically Reconciled: 3,840 rows

Evidence of the numeric variances our pipeline successfully forgave:


,amount_A,amount_B,variance
0,1933.73,1933.52,0.21
1,348217.27,348217.15,0.12
2,4630.35,4630.22,0.13
3,1913.00,1913.35,0.35
4,3361.66,3361.93,0.27


In [16]:
import pandas as pd
import numpy as np
import difflib

# 1. Define Paths
file_a = r'C:\Users\ASROCK\Desktop\Management\PROJECT\PROJECT 1 - AUTOMATED FINANCIAL PIPELINE\system-reconciliation-pipeline\data\raw\system_a_golden.csv'
file_b = r'C:\Users\ASROCK\Desktop\Management\PROJECT\PROJECT 1 - AUTOMATED FINANCIAL PIPELINE\system-reconciliation-pipeline\data\raw\system_b_messy.csv'
output_final_anomalies = r'C:\Users\ASROCK\Desktop\Management\PROJECT\PROJECT 1 - AUTOMATED FINANCIAL PIPELINE\system-reconciliation-pipeline\data\raw\final_anomalies_for_ML.csv'

print("🚀 INITIALIZING MASTER RECONCILIATION PIPELINE...\n")
df_a = pd.read_csv(file_a)
df_b = pd.read_csv(file_b)

total_records = len(df_a)

# --- TIER 1: EXACT MATCH ---
print("Step 1: Running Tier 1 (Exact Match Engine)...")
merged = pd.merge(df_a, df_b, how='outer', indicator=True)
exact_matches = merged[merged['_merge'] == 'both']
un_a = merged[merged['_merge'] == 'left_only'].drop(columns=['_merge']).copy()
un_b = merged[merged['_merge'] == 'right_only'].drop(columns=['_merge']).copy()

# --- TIER 2: FUZZY TEXT MATCHING ---
print("Step 2: Running Tier 2 (Fuzzy Text Resolution)...")
def get_similarity(str1, str2):
    return difflib.SequenceMatcher(None, str(str1), str(str2)).ratio()

potentials_text = pd.merge(un_a, un_b, on=['step', 'type', 'nameOrig', 'amount'], suffixes=('_A', '_B'))
potentials_text['sim_score'] = potentials_text.apply(lambda row: get_similarity(row['nameDest_A'], row['nameDest_B']), axis=1)
resolved_typos = potentials_text[potentials_text['sim_score'] >= 0.80]

# Filter out the resolved typos from our unmatched buckets
un_a = un_a[~un_a['nameDest'].isin(resolved_typos['nameDest_A'])]
un_b = un_b[~un_b['nameDest'].isin(resolved_typos['nameDest_B'])]

# --- TIER 3: NUMERIC TOLERANCE ---
print("Step 3: Running Tier 3 (Numeric Variance Engine)...")
potentials_num = pd.merge(un_a, un_b, on=['step', 'type', 'nameOrig', 'nameDest'], suffixes=('_A', '_B'))
potentials_num['variance'] = abs(potentials_num['amount_A'] - potentials_num['amount_B'])
resolved_amounts = potentials_num[(potentials_num['variance'] > 0) & (potentials_num['variance'] <= 0.50)]

# Filter out the resolved amounts from our unmatched buckets
un_a = un_a[~un_a['amount'].isin(resolved_amounts['amount_A'])]

# --- FINAL REPORT ---
print("\n=============================================")
print(" 📊 PIPELINE PERFORMANCE REPORT ")
print("=============================================")
print(f"Total Processed Records: {total_records:,}")
print(f"✅ Tier 1 (Exact):       {len(exact_matches):,}")
print(f"✅ Tier 2 (Typos):       {len(resolved_typos):,}")
print(f"✅ Tier 3 (Variances):   {len(resolved_amounts):,}")

total_reconciled = len(exact_matches) + len(resolved_typos) + len(resolved_amounts)
success_rate = (total_reconciled / total_records) * 100

print(f"\n🌟 OVERALL AUTOMATION SUCCESS RATE: {success_rate:.2f}%")
print(f"❌ True Anomalies Remaining: {len(un_a):,}")

# Save the true anomalies for Machine Learning (Phase 3)
un_a.to_csv(output_final_anomalies, index=False)
print(f"\nSaved true anomalies to: {output_final_anomalies}")

🚀 INITIALIZING MASTER RECONCILIATION PIPELINE...

Step 1: Running Tier 1 (Exact Match Engine)...
Step 2: Running Tier 2 (Fuzzy Text Resolution)...
Step 3: Running Tier 3 (Numeric Variance Engine)...

 📊 PIPELINE PERFORMANCE REPORT 
Total Processed Records: 200,000
✅ Tier 1 (Exact):       190,200
✅ Tier 2 (Typos):       3,701
✅ Tier 3 (Variances):   3,816

🌟 OVERALL AUTOMATION SUCCESS RATE: 98.86%
❌ True Anomalies Remaining: 2,244

Saved true anomalies to: C:\Users\ASROCK\Desktop\Management\PROJECT\PROJECT 1 - AUTOMATED FINANCIAL PIPELINE\system-reconciliation-pipeline\data\raw\final_anomalies_for_ML.csv


In [17]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler

# 1. Define Paths
file_anomalies = r'C:\Users\ASROCK\Desktop\Management\PROJECT\PROJECT 1 - AUTOMATED FINANCIAL PIPELINE\system-reconciliation-pipeline\data\raw\final_anomalies_for_ML.csv'
output_features = r'C:\Users\ASROCK\Desktop\Management\PROJECT\PROJECT 1 - AUTOMATED FINANCIAL PIPELINE\system-reconciliation-pipeline\data\raw\ml_features.csv'

print("🧠 Phase 3: Initializing Machine Learning Feature Engineering...")
df_ml = pd.read_csv(file_anomalies)

# 2. Select Core Features
# We drop names and IDs because they are unique to every row and don't help the AI find patterns.
# We just need the step (time), the type of transaction, and the amount.
df_features = df_ml[['step', 'type', 'amount']].copy()

# 3. Encode the Text (Categorical to Numerical)
print("Translating transaction types into machine-readable numbers...")
encoder = LabelEncoder()
# This turns 'CASH_IN', 'TRANSFER', etc., into numbers like 0, 1, 2.
df_features['type_encoded'] = encoder.fit_transform(df_features['type'])

# 4. Scale the Numbers
# AI gets confused if one column has tiny numbers (step = 1) and another has huge ones (amount = 500000).
# StandardScaler shrinks all the amounts down to a proportional scale centered around 0.
print("Standardizing the financial amounts...")
scaler = StandardScaler()
df_features['amount_scaled'] = scaler.fit_transform(df_features[['amount']])

# 5. Final Cleanup
# Drop the old text column and raw amount column so we are left with pure, scaled numbers.
df_features = df_features.drop(columns=['type', 'amount'])

# Save the ML-ready dataset
df_features.to_csv(output_features, index=False)

print("\n--- ⚙️ DAY 9: FEATURE ENGINEERING COMPLETE ---")
print(f"Prepared {len(df_features):,} records for the Isolation Forest algorithm.")
display(df_features.head())

🧠 Phase 3: Initializing Machine Learning Feature Engineering...
Translating transaction types into machine-readable numbers...
Standardizing the financial amounts...

--- ⚙️ DAY 9: FEATURE ENGINEERING COMPLETE ---
Prepared 2,244 records for the Isolation Forest algorithm.


,step,type_encoded,amount_scaled
0,5,3,-0.368145
1,6,3,-0.366191
2,7,1,-0.013833
3,7,3,-0.353801
4,8,0,-0.353733


In [18]:
import pandas as pd
from sklearn.ensemble import IsolationForest

# 1. Define Paths
features_file = r'C:\Users\ASROCK\Desktop\Management\PROJECT\PROJECT 1 - AUTOMATED FINANCIAL PIPELINE\system-reconciliation-pipeline\data\raw\ml_features.csv'
original_data_file = r'C:\Users\ASROCK\Desktop\Management\PROJECT\PROJECT 1 - AUTOMATED FINANCIAL PIPELINE\system-reconciliation-pipeline\data\raw\final_anomalies_for_ML.csv'
output_scored = r'C:\Users\ASROCK\Desktop\Management\PROJECT\PROJECT 1 - AUTOMATED FINANCIAL PIPELINE\system-reconciliation-pipeline\data\raw\scored_anomalies.csv'

print("🌲 Loading the ML environment...")
# Load the numerical features for the AI to read
df_features = pd.read_csv(features_file)
# Load the original text data so we can translate the AI's math back into human English
df_original = pd.read_csv(original_data_file)

# 2. Build the AI Model
# contamination=0.05 tells the AI: "Assume about 5% of these broken records are severe anomalies."
print("Training the Isolation Forest algorithm...")
model = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)

# 3. Predict & Score
print("Scoring the transactions...")
# The AI outputs 1 for "Normal" and -1 for "Anomaly"
df_original['anomaly_label'] = model.fit_predict(df_features)

# It also gives us a raw "suspicion score" (more negative = more suspicious)
df_original['anomaly_score'] = model.decision_function(df_features)

# 4. Isolate the Worst Offenders
# Filter down to just the records the AI flagged with a -1
severe_anomalies = df_original[df_original['anomaly_label'] == -1].copy()

# Sort them so the absolute most suspicious transaction is at the very top
severe_anomalies = severe_anomalies.sort_values(by='anomaly_score')

# Save the final risk report
severe_anomalies.to_csv(output_scored, index=False)

# 5. Print Results
print("\n--- 🚨 DAY 10: ANOMALY DETECTION COMPLETE ---")
print(f"The AI analyzed {len(df_features):,} unmatched records.")
print(f"It flagged {len(severe_anomalies)} transactions as HIGH RISK.")

print("\nHere are the Top 5 most suspicious transactions the AI found:")
display(severe_anomalies[['step', 'type', 'amount', 'nameOrig', 'nameDest', 'anomaly_score']].head())

🌲 Loading the ML environment...
Training the Isolation Forest algorithm...
Scoring the transactions...

--- 🚨 DAY 10: ANOMALY DETECTION COMPLETE ---
The AI analyzed 2,244 unmatched records.
It flagged 113 transactions as HIGH RISK.

Here are the Top 5 most suspicious transactions the AI found:


,step,type,amount,nameOrig,nameDest,anomaly_score
1431,300,TRANSFER,10895052.37,C358546790,C835309141,-0.180302
1777,355,TRANSFER,11973276.09,C1681640173,C1619434640,-0.178293
1581,326,TRANSFER,10432705.09,C1774602596,C218132038,-0.172322
1407,298,TRANSFER,5315954.72,C1787809986,C772629104,-0.152645
2212,614,TRANSFER,2739248.30,C1131884196,C97856418,-0.139610
